# Classical baselines on frozen protein embeddings

**Status:** drafting code for review.

**When to run:** after notebook 00; run once an encoder cache is complete.

**Inputs:** shared split CSVs and one or more existing `.pt` embedding
caches under `embeddings/`.

**Outputs:** optional per-seed metrics and row-level test predictions under
`tables/drafting_code`.

**Safety:** no cache or split is modified. `WRITE_OUTPUTS=False` by default.
Start with `ENCODERS_TO_RUN=["ProtT5"]`; add encoders only after verifying
their cache metadata and split hashes. `QUICK_MODE=True` runs one seed on a
bounded training subset; turn it off for the five-seed reporting run.

The validation split selects the threshold. The test labels are used once
for final reporting. Dummy, logistic regression, random forest, ExtraTrees,
and histogram gradient boosting provide increasingly nonlinear baselines.

## 1. Setup and run controls

In [ ]:
from pathlib import Path
import os
import sys

# Keep the repository path separate from SNOOPPI_ROOT, which stores data/artifacts.
# In Colab the recommended clone location is /content/SNOOPPI.
DRAFT_DIR = Path.cwd()
if not (DRAFT_DIR / "snooppi_utils.py").exists():
    repository_root = Path(os.environ.get("SNOOPPI_REPO_ROOT", "/content/SNOOPPI"))
    candidate = repository_root / "drafting_code"
    if candidate.exists():
        DRAFT_DIR = candidate
if not (DRAFT_DIR / "snooppi_utils.py").exists():
    raise FileNotFoundError(
        "Cannot locate drafting_code/snooppi_utils.py. Open the notebook from the "
        "repository clone or set SNOOPPI_REPO_ROOT to the clone root."
    )
if str(DRAFT_DIR) not in sys.path:
    sys.path.insert(0, str(DRAFT_DIR))

from snooppi_utils import (
    LABEL,
    SEQUENCE_A,
    SEQUENCE_B,
    canonical_pair_id,
    ensure_output_directories,
    experiment_paths,
    load_splits,
    normalize_sequence,
    sequence_id,
    split_manifest,
)

PATHS = experiment_paths()
ensure_output_directories(PATHS)
print("SNOOPPI root:", PATHS["root"])

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from snooppi_utils import (
    build_pair_features,
    load_embedding_cache,
    select_threshold,
    summarize_predictions,
)

RUN_TAG = "classical_baselines_v1"
WRITE_OUTPUTS = False
ENCODERS_TO_RUN = ["ProtT5"]
QUICK_MODE = True

# QUICK_MODE is the safe usage default: one seed, three representative models,
# and a bounded training subset. Disable it before a reported experiment.
REPORTING_MODELS = ["dummy", "logistic", "random_forest", "extra_trees", "hist_gradient_boosting"]
REPORTING_SEEDS = [41, 42, 43, 44, 45]
MODEL_NAMES_TO_RUN = ["dummy", "logistic", "random_forest"] if QUICK_MODE else REPORTING_MODELS
SEEDS = [44] if QUICK_MODE else REPORTING_SEEDS
MAX_TRAIN_ROWS = 5_000 if QUICK_MODE else None

CACHE_PATHS = {
    "ProtT5": PATHS["embeddings"] / "prot_t5_embedding_cache.pt",
    "ESM-2 650M": PATHS["embeddings"] / "esm2_t33_650M_snooppi_mean_pool_embeddings.pt",
    "ESM-C 600M": PATHS["embeddings"] / "esmc_600m_snooppi_mean_pool_embeddings.pt",
}

## 2. Load inputs and verify identities

In [ ]:
splits = load_splits(PATHS)
manifest = split_manifest(PATHS)
print(pd.Series(manifest["sha256"], name="split_sha256"))

missing_caches = [
    name for name in ENCODERS_TO_RUN if not CACHE_PATHS[name].exists()
]
if missing_caches:
    raise FileNotFoundError(
        "Build or copy the requested encoder caches first: " + ", ".join(missing_caches)
    )

train_frame = splits["train"]
if MAX_TRAIN_ROWS is not None:
    # Development-only subsampling preserves class balance and never changes val/test.
    train_frame = (
        train_frame.groupby(LABEL, group_keys=False)
        .apply(lambda group: group.sample(
            n=min(len(group), MAX_TRAIN_ROWS // 2), random_state=44
        ))
        .sample(frac=1, random_state=44)
        .reset_index(drop=True)
    )
    print("Development subset rows:", len(train_frame))

## 3. Define models and probability extraction

In [ ]:
def model_registry(seed):
    """Fresh estimators per seed prevent fitted-state leakage between runs."""
    return {
        "dummy": DummyClassifier(strategy="prior"),
        "logistic": Pipeline([
            ("scale", StandardScaler()),
            ("classifier", LogisticRegression(
                C=1.0,
                max_iter=3_000,
                class_weight="balanced",
                random_state=seed,
            )),
        ]),
        "random_forest": RandomForestClassifier(
            n_estimators=400,
            max_features="sqrt",
            min_samples_leaf=2,
            class_weight="balanced_subsample",
            n_jobs=-1,
            random_state=seed,
        ),
        "extra_trees": ExtraTreesClassifier(
            n_estimators=400,
            max_features="sqrt",
            min_samples_leaf=2,
            class_weight="balanced",
            n_jobs=-1,
            random_state=seed,
        ),
        "hist_gradient_boosting": HistGradientBoostingClassifier(
            learning_rate=0.05,
            max_iter=300,
            max_leaf_nodes=31,
            l2_regularization=1e-3,
            early_stopping=True,
            random_state=seed,
        ),
    }


def positive_probability(model, features):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(features)[:, 1]
    decision = model.decision_function(features)
    return 1.0 / (1.0 + np.exp(-decision))

## 4. Fit repeated-seed baselines

In [ ]:
metric_rows = []
prediction_rows = []

for encoder_name in ENCODERS_TO_RUN:
    cache, cache_metadata = load_embedding_cache(CACHE_PATHS[encoder_name])
    print(f"{encoder_name}: {len(cache):,} cached proteins; metadata:", cache_metadata)

    # Use the same symmetric representation as the preserved neural classifiers.
    x_train = build_pair_features(train_frame, cache)
    x_validation = build_pair_features(splits["validation"], cache)
    x_test = build_pair_features(splits["test"], cache)
    y_train = train_frame[LABEL].to_numpy()
    y_validation = splits["validation"][LABEL].to_numpy()
    y_test = splits["test"][LABEL].to_numpy()

    for seed in SEEDS:
        registry = model_registry(seed)
        for model_name in MODEL_NAMES_TO_RUN:
            started = time.perf_counter()
            model = registry[model_name]
            model.fit(x_train, y_train)
            validation_probability = positive_probability(model, x_validation)
            threshold = select_threshold(y_validation, validation_probability)
            test_probability = positive_probability(model, x_test)
            metrics = summarize_predictions(y_test, test_probability, threshold)
            metrics.update({
                "encoder": encoder_name,
                "model": model_name,
                "seed": seed,
                "fit_seconds": time.perf_counter() - started,
                "feature_dimension": x_train.shape[1],
                "development_subset": QUICK_MODE,
            })
            metric_rows.append(metrics)

            for row_index, (label, probability) in enumerate(zip(y_test, test_probability)):
                prediction_rows.append({
                    "encoder": encoder_name,
                    "model": model_name,
                    "seed": seed,
                    "row_index": row_index,
                    "label": int(label),
                    "probability": float(probability),
                    "threshold": threshold,
                })
            print(encoder_name, model_name, seed, f"macro-AP={metrics['macro_ap']:.4f}")

metrics_table = pd.DataFrame(metric_rows)
predictions_table = pd.DataFrame(prediction_rows)
display(metrics_table.sort_values("macro_ap", ascending=False).head(20).round(4))

## 5. Compare mean performance and seed variability

In [ ]:
comparison = (
    metrics_table.groupby(["encoder", "model"])
    .agg(
        macro_ap_mean=("macro_ap", "mean"),
        macro_ap_sd=("macro_ap", "std"),
        mcc_mean=("mcc", "mean"),
        mcc_sd=("mcc", "std"),
        auroc_mean=("auroc", "mean"),
        brier_mean=("brier", "mean"),
        seeds=("seed", "nunique"),
    )
    .reset_index()
    .sort_values("macro_ap_mean", ascending=True)
)
display(comparison.round(4))

labels = comparison["encoder"] + " · " + comparison["model"]
figure, axis = plt.subplots(figsize=(9, max(4.5, 0.42 * len(comparison))))
axis.barh(labels, comparison["macro_ap_mean"], xerr=comparison["macro_ap_sd"], color="#176B87", alpha=0.9)
axis.set(
    title="Classical baselines on frozen symmetric pair features",
    xlabel="Test macro average precision (mean ± SD across seeds)",
    ylabel="",
    xlim=(0, 1),
)
axis.grid(axis="x", alpha=0.2)
axis.grid(axis="y", visible=False)
figure.tight_layout()
plt.show()

## 6. Optional outputs

In [ ]:
if WRITE_OUTPUTS:
    metrics_path = PATHS["tables"] / f"{RUN_TAG}_metrics.csv"
    predictions_path = PATHS["tables"] / f"{RUN_TAG}_test_predictions.csv"
    comparison_path = PATHS["tables"] / f"{RUN_TAG}_summary.csv"
    figure_path = PATHS["figures"] / f"{RUN_TAG}_macro_ap.png"
    for path in (metrics_path, predictions_path, comparison_path, figure_path):
        if path.exists():
            raise FileExistsError(f"Refusing to overwrite {path}; change RUN_TAG")
    metrics_table.to_csv(metrics_path, index=False)
    predictions_table.to_csv(predictions_path, index=False)
    comparison.to_csv(comparison_path, index=False)
    figure.savefig(figure_path, dpi=300, bbox_inches="tight")
    print("Saved:", metrics_path.parent)
else:
    print("Preview only. Choose a unique RUN_TAG and set WRITE_OUTPUTS=True to save.")

## Takeaways to write after execution

Report macro-AP, positive AP, MCC, Brier score, seed variability, class
prevalence, and fit time. Do not select the winning method on test values;
if model/hyperparameter selection expands, use validation or nested CV and
reserve the test set for one final comparison.